In [9]:
"""
Encoder–Decoder with Attention — From Scratch (Tutorial Script)

Pre-requisite: You've already learned Word2Vec (word embeddings). Now we move to the next big
building block of modern LLMs: the Sequence-to-Sequence (Seq2Seq) architecture with Attention.

Sections:
1. Why do we need Encoder-Decoder?
2. The Encoder — Reading the Input
3. The Decoder — Generating the Output
4. The Bottleneck Problem
5. Attention Mechanism — The Breakthrough
6. Math Deep-Dive
7. Hands-On: Building & Training from Scratch
8. Visualizing Attention
9. Exercises
"""

import sys
import os

In [10]:
import numpy as np
np.random.seed(42)

from encoder_decoder_attention.encoder_decoder_scratch import (
    Vocabulary, GRUCell, EmbeddingLayer, BahdanauAttention,
    LinearLayer, Seq2SeqAttention, softmax,
    plot_attention_matrix, print_attention_text
)

print("All modules imported successfully! [OK]")


All modules imported successfully! [OK]


In [11]:
# =============================================================================
# 7.1 Understanding Each Building Block
# =============================================================================

# ---- A) Vocabulary — Mapping words <-> numbers ----
print("\n" + "=" * 60)
print("A) VOCABULARY")
print("=" * 60)

src_sentences = [
    ["i", "love", "cats"],
    ["cats", "are", "cute"],
    ["i", "like", "dogs"],
]

vocab = Vocabulary()
vocab.build_from_sentences(src_sentences)

print("\nWord -> Index mapping:")
for word, idx in sorted(vocab.word2idx.items(), key=lambda x: x[1]):
    print(f"  '{word}' -> {idx}")

encoded = vocab.encode(["i", "love", "cats"])
print(f"\nEncoded 'i love cats': {encoded}")
print(f"Decoded back: {vocab.decode(encoded)}")


# ---- B) Embedding Layer — From indices to dense vectors ----
print("\n" + "=" * 60)
print("B) EMBEDDING LAYER")
print("=" * 60)

emb_layer = EmbeddingLayer(vocab_size=10, embedding_dim=4)

vec = emb_layer.forward(2)
print(f"Embedding for index 2:")
print(f"  Shape: {vec.shape}")
print(f"  Values: {vec.flatten()}")

print(f"\nFull embedding matrix shape: {emb_layer.weights.shape}")
print("(Each row = one word's vector)")


# ---- C) GRU Cell — Processing one time-step ----
print("\n" + "=" * 60)
print("C) GRU CELL")
print("=" * 60)

gru = GRUCell(input_dim=4, hidden_dim=8)

h = np.zeros((8, 1))
print(f"Initial hidden state shape: {h.shape}")
print(f"Initial hidden state (all zeros): {h.flatten()[:4]}...")

words = ["word_1", "word_2", "word_3"]
for i, word in enumerate(words):
    x = np.random.randn(4, 1) * 0.1
    h = gru.forward(x, h)
    print(f"\nAfter '{word}':")
    print(f"  Hidden state: [{h[0,0]:.4f}, {h[1,0]:.4f}, {h[2,0]:.4f}, ...]")
    print(f"  (State captures all words seen so far!)")


# ---- D) Attention — The core mechanism ----
print("\n" + "=" * 60)
print("D) ATTENTION MECHANISM")
print("=" * 60)

hidden_dim = 8
attn = BahdanauAttention(encoder_dim=hidden_dim, decoder_dim=hidden_dim, attention_dim=6)

encoder_outputs = [
    np.random.randn(hidden_dim, 1) * 0.5,  # h_1 (for "I")
    np.random.randn(hidden_dim, 1) * 0.5,  # h_2 (for "love")
    np.random.randn(hidden_dim, 1) * 0.5,  # h_3 (for "cats")
]

decoder_hidden = np.random.randn(hidden_dim, 1) * 0.5

context, weights = attn.forward(decoder_hidden, encoder_outputs)

src_words = ["I", "love", "cats"]
print("Attention weights (which source words to focus on):")
for word, w in zip(src_words, weights):
    bar = "#" * int(w * 40)
    print(f"  {word:>6s}: {w:.4f}  {bar}")

print(f"\nWeights sum to: {weights.sum():.4f} (should be 1.0)")
print(f"Context vector shape: {context.shape} (weighted mix of encoder states)")



A) VOCABULARY
Vocabulary built: 11 tokens (including 4 special tokens)

Word -> Index mapping:
  '<pad>' -> 0
  '<sos>' -> 1
  '<eos>' -> 2
  '<unk>' -> 3
  'i' -> 4
  'love' -> 5
  'cats' -> 6
  'are' -> 7
  'cute' -> 8
  'like' -> 9
  'dogs' -> 10

Encoded 'i love cats': [4, 5, 6]
Decoded back: ['i', 'love', 'cats']

B) EMBEDDING LAYER
Embedding for index 2:
  Shape: (4, 1)
  Values: [-0.04694744  0.054256   -0.04634177 -0.04657298]

Full embedding matrix shape: (10, 4)
(Each row = one word's vector)

C) GRU CELL
Initial hidden state shape: (8, 1)
Initial hidden state (all zeros): [0. 0. 0. 0.]...

After 'word_1':
  Hidden state: [-0.0278, 0.0487, 0.0065, ...]
  (State captures all words seen so far!)

After 'word_2':
  Hidden state: [-0.0147, -0.0218, 0.0510, ...]
  (State captures all words seen so far!)

After 'word_3':
  Hidden state: [0.0097, -0.0600, 0.0624, ...]
  (State captures all words seen so far!)

D) ATTENTION MECHANISM
Attention weights (which source words to focus on

In [12]:
# =============================================================================
# 7.2 Training a Full Seq2Seq Model
# =============================================================================
print("\n" + "=" * 60)
print("TRAINING A FULL SEQ2SEQ MODEL")
print("=" * 60)

training_pairs = [
    (["one", "two", "three"], ["un", "deux", "trois"]),
    (["four", "five"],        ["quatre", "cinq"]),
    (["one", "three"],        ["un", "trois"]),
    (["two", "four", "five"], ["deux", "quatre", "cinq"]),
]

# Build source & target vocabularies
src_vocab = Vocabulary()
tgt_vocab = Vocabulary()

src_sentences = [pair[0] for pair in training_pairs]
tgt_sentences = [pair[1] for pair in training_pairs]

src_vocab.build_from_sentences(src_sentences)
tgt_vocab.build_from_sentences(tgt_sentences)

print(f"\nSource vocabulary: {src_vocab.word2idx}")
print(f"Target vocabulary: {tgt_vocab.word2idx}")

# Prepare training data
sos_idx = tgt_vocab.word2idx["<sos>"]
eos_idx = tgt_vocab.word2idx["<eos>"]

data = []
for src_sent, tgt_sent in training_pairs:
    src_ids = src_vocab.encode(src_sent)
    tgt_ids = [sos_idx] + tgt_vocab.encode(tgt_sent) + [eos_idx]
    data.append((src_ids, tgt_ids))

print("\nPrepared training examples:")
for i, (s, t) in enumerate(data):
    src_words = src_vocab.decode(s)
    tgt_words = tgt_vocab.decode(t)
    print(f"  {i}: {src_words} -> {tgt_words}")
    print(f"     {s} -> {t}")

# Create model
np.random.seed(42)

model = Seq2SeqAttention(
    src_vocab_size=src_vocab.n_words,
    tgt_vocab_size=tgt_vocab.n_words,
    embedding_dim=16,
    hidden_dim=32,
    attention_dim=16,
    learning_rate=0.05
)

print("\nModel created!")
print(f"  Encoder embedding: {model.enc_embedding.weights.shape}")
print(f"  Decoder embedding: {model.dec_embedding.weights.shape}")
print(f"  Hidden dim: {model.hidden_dim}")
print(f"  Output layer: {model.output_layer.W.shape}")

# Training loop
print("\nTraining (numerical gradients — be patient)...\n")

num_epochs = 15
for epoch in range(num_epochs):
    total_loss = 0
    for src_ids, tgt_ids in data:
        loss = model.train_step(src_ids, tgt_ids)
        total_loss += loss

    avg_loss = total_loss / len(data)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>3d}/{num_epochs}  |  Avg Loss: {avg_loss:.4f}")

print("\nTraining complete! [OK]")



TRAINING A FULL SEQ2SEQ MODEL
Vocabulary built: 9 tokens (including 4 special tokens)
Vocabulary built: 9 tokens (including 4 special tokens)

Source vocabulary: {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'one': 4, 'two': 5, 'three': 6, 'four': 7, 'five': 8}
Target vocabulary: {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'un': 4, 'deux': 5, 'trois': 6, 'quatre': 7, 'cinq': 8}

Prepared training examples:
  0: ['one', 'two', 'three'] -> ['<sos>', 'un', 'deux', 'trois', '<eos>']
     [4, 5, 6] -> [1, 4, 5, 6, 2]
  1: ['four', 'five'] -> ['<sos>', 'quatre', 'cinq', '<eos>']
     [7, 8] -> [1, 7, 8, 2]
  2: ['one', 'three'] -> ['<sos>', 'un', 'trois', '<eos>']
     [4, 6] -> [1, 4, 6, 2]
  3: ['two', 'four', 'five'] -> ['<sos>', 'deux', 'quatre', 'cinq', '<eos>']
     [5, 7, 8] -> [1, 5, 7, 8, 2]

Model created!
  Encoder embedding: (9, 16)
  Decoder embedding: (9, 16)
  Hidden dim: 32
  Output layer: (9, 32)

Training (numerical gradients — be patient)...

Epoch   1/15  |  Avg 

In [13]:

# =============================================================================
# 7.3 Testing the Model — Greedy Decoding
# =============================================================================
print("\n" + "=" * 50)
print("TRANSLATION RESULTS")
print("=" * 50)

for src_sent, expected in training_pairs:
    src_ids = src_vocab.encode(src_sent)
    output_ids, attn_matrix = model.translate(
        src_ids, max_len=10, sos_idx=sos_idx, eos_idx=eos_idx
    )
    output_words = tgt_vocab.decode(output_ids)

    output_clean = [w for w in output_words if w != "<eos>"]

    print(f"  Input:    {' '.join(src_sent)}")
    print(f"  Expected: {' '.join(expected)}")
    print(f"  Got:      {' '.join(output_clean)}")
    print()


# =============================================================================
# 8. Visualizing Attention
# =============================================================================
print("\n" + "=" * 60)
print("VISUALIZING ATTENTION")
print("=" * 60)

src_sent = training_pairs[0][0]
src_ids = src_vocab.encode(src_sent)

output_ids, attn_matrix = model.translate(
    src_ids, max_len=10, sos_idx=sos_idx, eos_idx=eos_idx
)
output_words = tgt_vocab.decode(output_ids)

print(f"Source:  {src_sent}")
print(f"Output:  {output_words}")

# Text-based attention heatmap
print_attention_text(attn_matrix, src_sent, output_words)

# Graphical attention heatmap (if matplotlib is installed)
try:
    plot_attention_matrix(attn_matrix, src_sent, output_words)
except Exception as e:
    print(f"(Could not plot: {e})")
    print("Install matplotlib for visual plots: pip install matplotlib")



TRANSLATION RESULTS
  Input:    one two three
  Expected: un deux trois
  Got:      

  Input:    four five
  Expected: quatre cinq
  Got:      

  Input:    one three
  Expected: un trois
  Got:      

  Input:    two four five
  Expected: deux quatre cinq
  Got:      


VISUALIZING ATTENTION
Source:  ['one', 'two', 'three']
Output:  ['<eos>']

  Attention Heatmap (rows=target, cols=source)
                one       two     three
           ------------------------------
     <eos> |    0.299     0.374     0.328

matplotlib not installed — skipping plot


In [14]:
# =============================================================================
# Step-by-step Forward Pass Walkthrough
# =============================================================================
print("\n" + "=" * 60)
print("STEP-BY-STEP FORWARD PASS")
print("=" * 60)

src = ["one", "two", "three"]
tgt = ["un", "deux", "trois"]

src_ids = src_vocab.encode(src)
tgt_ids = [sos_idx] + tgt_vocab.encode(tgt) + [eos_idx]

print(f"\nSource: {src}  ->  indices: {src_ids}")
print(f"Target: ['<sos>'] + {tgt} + ['<eos>']  ->  indices: {tgt_ids}")

# --- ENCODING ---
print("\n--- ENCODING ---")
encoder_outputs, enc_final = model.encode(src_ids)
print(f"Processed {len(src_ids)} source tokens through encoder GRU")
for i, (word, h) in enumerate(zip(src, encoder_outputs)):
    print(f"  h_{i+1} ('{word}'): [{h[0,0]:.4f}, {h[1,0]:.4f}, ...] shape={h.shape}")
print(f"Encoder final state -> Decoder initial state: shape={enc_final.shape}")

# --- DECODING ---
print("\n--- DECODING (with attention) ---")
h_dec = enc_final.copy()

for t in range(len(tgt_ids) - 1):
    input_tok = tgt_ids[t]
    label = tgt_ids[t + 1]
    input_word = tgt_vocab.idx2word[input_tok]
    label_word = tgt_vocab.idx2word[label]

    logits, h_dec, attn_w = model.decode_step(input_tok, h_dec, encoder_outputs)
    probs = softmax(logits.flatten())
    predicted_idx = int(np.argmax(probs))
    predicted_word = tgt_vocab.idx2word[predicted_idx]

    print(f"\n  Step {t+1}:")
    print(f"    Input token: '{input_word}' (idx={input_tok})")
    print(f"    Attention weights: {dict(zip(src, [f'{w:.3f}' for w in attn_w]))}")
    print(f"    Predicted: '{predicted_word}' (prob={probs[predicted_idx]:.4f})")
    print(f"    Correct:   '{label_word}'")
    print(f"    {'[OK]' if predicted_idx == label else '[X]'}")




STEP-BY-STEP FORWARD PASS

Source: ['one', 'two', 'three']  ->  indices: [4, 5, 6]
Target: ['<sos>'] + ['un', 'deux', 'trois'] + ['<eos>']  ->  indices: [1, 4, 5, 6, 2]

--- ENCODING ---
Processed 3 source tokens through encoder GRU
  h_1 ('one'): [-0.0460, 0.0092, ...] shape=(32, 1)
  h_2 ('two'): [-0.0227, 0.0222, ...] shape=(32, 1)
  h_3 ('three'): [0.0132, 0.1273, ...] shape=(32, 1)
Encoder final state -> Decoder initial state: shape=(32, 1)

--- DECODING (with attention) ---

  Step 1:
    Input token: '<sos>' (idx=1)
    Attention weights: {'one': '0.299', 'two': '0.374', 'three': '0.328'}
    Predicted: '<eos>' (prob=0.2072)
    Correct:   'un'
    [X]

  Step 2:
    Input token: 'un' (idx=4)
    Attention weights: {'one': '0.298', 'two': '0.370', 'three': '0.332'}
    Predicted: '<eos>' (prob=0.2988)
    Correct:   'deux'
    [X]

  Step 3:
    Input token: 'deux' (idx=5)
    Attention weights: {'one': '0.304', 'two': '0.363', 'three': '0.333'}
    Predicted: '<eos>' (prob=0.3